In [5]:
with open("../data/clean_logs.txt") as f:
    logs = [line.strip() for line in f]

# print(logs[:100])
update_list=list(set(logs))
print(update_list)

['ip Exception writing block to mirror', 'writeBlock block received exception millis timeout while waiting for channel to be ready read. ch [connected local=path remote=path]', 'Receiving block source path destination', 'PacketResponder block Exception Connection reset by peer', 'PacketResponder block Exception Interruped while waiting for IO on channel [connected local=path remote=path]. millis timeout left.', 'BLOCK Removing block from neededReplications as it does not belong to any file.', 'Deleting block file path', 'Changing block file offset of from to meta', 'writeBlock block received exception No route to host', 'Received block of size from path', 'writeBlock block received exception Broken pipe', 'PacketResponder block Exception Broken pipe', 'Exception in receiveBlock for block Interruped while waiting IO on channel [connected local=path remote=path]. millis timeout left.', 'writeBlock block received exception Block is valid, and cannot be written to.', 'writeBlock block rece

In [6]:
# clean_logs = list(set(logs))
len(update_list)

49

In [8]:
import csv

# logs = ["Error 1", "Error 2", "Error 3"]

import pandas as pd

# If you have a list
df = pd.DataFrame(update_list, columns=["Log Name"])

# Save to CSV
df.to_csv('simple_list.csv', index=False)

In [9]:
from sklearn.cluster import DBSCAN
from sentence_transformers import SentenceTransformer

In [11]:
model = SentenceTransformer('all-MiniLM-L6-v2')  # Lightweight embedding model
embeddings = model.encode(df['Log Name'].tolist())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
embeddings[:5]

array([[ 0.02916159,  0.02136257,  0.0700311 , ...,  0.07112961,
         0.00031429, -0.02258485],
       [ 0.08344477, -0.02922634, -0.04404536, ...,  0.00997415,
        -0.10194498,  0.04911781],
       [-0.01213467, -0.01590658, -0.06338082, ...,  0.07024688,
        -0.00826929,  0.02524849],
       [-0.05682135, -0.00302603,  0.04457267, ...,  0.06644502,
        -0.06521035, -0.02055228],
       [ 0.00917645, -0.00743421, -0.00704707, ...,  0.0027688 ,
        -0.13814446, -0.00410257]], shape=(5, 384), dtype=float32)

In [13]:
clustering = DBSCAN(eps=0.2, min_samples=1, metric='cosine').fit(embeddings)
df['cluster'] = clustering.labels_

In [15]:
df.head(49)

,Log Name,cluster
0,ip Exception writing block to mirror,0
1,writeBlock block received exception millis tim...,1
2,Receiving block source path destination,2
3,PacketResponder block Exception Connection res...,3
4,PacketResponder block Exception Interruped whi...,1
5,BLOCK Removing block from neededReplications a...,4
6,Deleting block file path,5
7,Changing block file offset of from to meta,6
8,writeBlock block received exception No route t...,7
9,Received block of size from path,2


In [19]:
# Group by cluster to inspect patterns
clusters = df.groupby('cluster')['Log Name'].apply(list)
sorted_clusters = clusters.sort_values(key=lambda x: x.map(len), ascending=False)
print(sorted_clusters)

cluster
7     [writeBlock block received exception No route ...
1     [writeBlock block received exception millis ti...
3     [PacketResponder block Exception Connection re...
2     [Receiving block source path destination, Rece...
16    [ip Served block to path, ip Transmitted block...
13    [BLOCK Redundant addStoredBlock request receiv...
17    [ip Starting thread to transfer block ip,, ip ...
0                [ip Exception writing block to mirror]
20           [BLOCK block is added to invalidSet of ip]
19    [Unexpected error trying to delete block block...
18    [ip Failed to transfer block got Connection re...
15                   [Adding an already existing block]
14                                   [BLOCK path block]
11                                 [Reopen Block block]
12                   [Receiving empty packet for block]
10                   [Verification succeeded for block]
9           [PendingReplicationMonitor timed out block]
8         [BLOCK ask ip to replicate blo

In [22]:
print("Clustered Patterns:")
for cluster_id, messages in sorted_clusters.items():
    # if len(messages) > 10:
    print(f"Cluster {cluster_id}:")
    for msg in messages[:25]:
        print(f"  {msg}")

Clustered Patterns:
Cluster 7:
  writeBlock block received exception No route to host
  writeBlock block received exception Broken pipe
  writeBlock block received exception Block is valid, and cannot be written to.
  writeBlock block received exception Interrupted receiveBlock
  Exception in receiveBlock for block
  Exception in receiveBlock for block Broken pipe
  writeBlock block received exception Could not read from stream
  writeBlock block received exception Connection reset by peer
  Exception in receiveBlock for block Connection reset by peer
  writeBlock block received exception
Cluster 1:
  writeBlock block received exception millis timeout while waiting for channel to be ready read. ch [connected local=path remote=path]
  PacketResponder block Exception Interruped while waiting for IO on channel [connected local=path remote=path]. millis timeout left.
  Exception in receiveBlock for block Interruped while waiting IO on channel [connected local=path remote=path]. millis time